# Data Cleansing

- Raw data:
  - read only
  - source `data/raw/<year>/*.csv`
- Staging data:
  - source `data/staging/<year>/*.csv`
- Cleaned data:
  - source `data/cleaned/<year>/*.csv`


## Schema check

- Inspect schema via columns name, data type


In [15]:
# schema info
from pathlib import Path
import pandas as pd

# RAW_DIR = Path("../data/raw")
RAW_DIR = Path("../data/raw")

# read a small sample per file for dtypes
SAMPLE_ROWS = 50

records = []
for path in sorted(RAW_DIR.glob("*/*.csv")):
    sample = pd.read_csv(path, nrows=SAMPLE_ROWS, low_memory=False, encoding='ansi')

    for position, column in enumerate(sample.columns):
        records.append(
            {
                "year": path.parent.name,
                "file": path.name,
                "position": position,
                "column": column,
                "dtype": str(sample[column].dtype),
            }
        )

schema = pd.DataFrame(records)
print(f"{schema['file'].nunique()} files, {schema['column'].nunique()} distinct column names")

schema

64 files, 22 distinct column names


,year,file,position,column,dtype
0,2019,2019-Q1.csv,0,ï»¿Trip Id,int64
1,2019,2019-Q1.csv,1,Trip Duration,int64
2,2019,2019-Q1.csv,2,Start Station Id,int64
3,2019,2019-Q1.csv,3,Start Time,str
4,2019,2019-Q1.csv,4,Start Station Name,str
...,...,...,...,...,...
648,2025,bikeshare_2025_12.csv,6,End_Time,str
649,2025,bikeshare_2025_12.csv,7,End_Station_Name,str
650,2025,bikeshare_2025_12.csv,8,Bike_Id,int64
651,2025,bikeshare_2025_12.csv,9,User_Type,str


### File naming

- 2019: 4 quarterly files (`2019-Q1.csv`)
- 2020: 12 monthly (`2020-01.csv`)
- 2021–2023: 12 monthly (`Bike share ridership 2021-01.csv` — spaces in filename)
- 2024: **1 annual file** (`bikeshare-ridership-2024.csv`)
- 2025: 12 monthly (`bikeshare_2025_01.csv`)

---

### Shema shift

Two schema eras across the 64 raw files, with the break between 2023 and 2024.

|                    | Era A     | Era B      |
| ------------------ | --------- | ---------- |
| **Years**          | 2019–2023 | 2024–2025  |
| **Files**          | 51        | 13         |
| **Columns**        | 10        | 11         |
| **Name separator** | space     | underscore |

---

### Normalized Shema

| #   | Era A                             | Era B                | Normalized           | Dtype                   |
| --- | --------------------------------- | -------------------- | -------------------- | ----------------------- |
| 0   | `Trip Id`                         | `Trip_Id`            | `trip_id`            | `Int64`                 |
| 1   | `Trip  Duration` _(double space)_ | `Trip_Duration`      | `trip_duration`      | `Int64`                 |
| 2   | `Start Station Id`                | `Start_Station_Id`   | `start_station_id`   | `Int64`                 |
| 3   | `Start Time`                      | `Start_Time`         | `start_time`         | `datetime64[ns]`        |
| 4   | `Start Station Name`              | `Start_Station_Name` | `start_station_name` | `string`                |
| 5   | `End Station Id`                  | `End_Station_Id`     | `end_station_id`     | `Int64` _(nullable)_    |
| 6   | `End Time`                        | `End_Time`           | `end_time`           | `datetime64[ns]`        |
| 7   | `End Station Name`                | `End_Station_Name`   | `end_station_name`   | `string` _(nullable)_   |
| 8   | `Bike Id`                         | `Bike_Id`            | `bike_id`            | `Int64`                 |
| 9   | `User Type`                       | `User_Type`          | `user_type`          | `category`              |
| 10  | —                                 | `Bike_Model`         | `bike_model`         | `category` _(nullable)_ |

---


## Validity check

- path: `staging/<year>/*.csv`
- Normalize Value formats per column


In [ ]:
# Normalize Header
# trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model
import re
from pathlib import Path

import pandas as pd

STAGING_DIR = Path("../data/staging")

# canonical order from the Normalized Schema table
NORMALIZED_COLUMNS = [
    "trip_id",
    "trip_duration",
    "start_station_id",
    "start_time",
    "start_station_name",
    "end_station_id",
    "end_time",
    "end_station_name",
    "bike_id",
    "user_type",
    "bike_model",
]


def detect_encoding(path: Path) -> str:
    """Staging is mixed: utf-8 (with or without BOM) and cp1252.

    utf-8-sig strips the BOM when present and is a no-op when absent, so it
    covers both utf-8 cases; cp1252 is the fallback for the rest.
    """
    raw = path.read_bytes()
    try:
        raw.decode("utf-8")
        return "utf-8-sig"
    except UnicodeDecodeError:
        return "cp1252"


def normalize_header(name: str) -> str:
    """Era A ('Trip  Duration') and Era B ('Trip_Duration') -> 'trip_duration'."""
    name = name.replace("﻿", "").strip()   # stray BOM if decoded as cp1252
    name = re.sub(r"[\s_]+", "_", name)         # collapse spaces/underscores
    return name.lower()


report = []
for path in sorted(STAGING_DIR.glob("*/*.csv")):
    encoding = detect_encoding(path)
    df = pd.read_csv(path, dtype=str, low_memory=False, encoding=encoding)

    original = list(df.columns)
    df.columns = [normalize_header(c) for c in df.columns]

    unexpected = [c for c in df.columns if c not in NORMALIZED_COLUMNS]
    if unexpected:
        raise ValueError(f"{path.name}: unexpected column(s) {unexpected}")

    # keep canonical order; Era A files simply lack bike_model
    df = df[[c for c in NORMALIZED_COLUMNS if c in df.columns]]

    # rewrite in place as utf-8, no BOM
    df.to_csv(path, index=False, encoding="utf-8")

    report.append(
        {
            "year": path.parent.name,
            "file": path.name,
            "read_as": encoding,
            "n_columns": len(df.columns),
            "renamed": sum(o != n for o, n in zip(original, df.columns)),
        }
    )

report = pd.DataFrame(report)
print(f"{len(report)} files normalized")
print(report["read_as"].value_counts().to_string())
report

In [ ]:
# confirm normalized
headers = {}
for path in sorted(STAGING_DIR.glob("*/*.csv")):
    with open(path, encoding="utf-8") as fh:
        headers.setdefault(fh.readline().strip(), []).append(f"{path.parent.name}/{path.name}")

for header, files in headers.items():
    print(f"{len(files):>3} files: {header}")

assert all(
    h.split(",") == NORMALIZED_COLUMNS[: len(h.split(","))] for h in headers
), "header not in canonical order"
print("\nAll headers normalized and in canonical order.")

## Completeness Check

- path: `staging/<year>/*.csv`
- Missing or null values per column


## Uniqueness Check

- path: `staging/<year>/*.csv`
- Duplicate records or keys


## Consistency Check

- path: `staging/<year>/*.csv`
- Related fields do not conflict


## Timeliness Check

- path: `staging/<year>/*.csv`
- Data is recent enough
